In [80]:
import json
import pandas as pd
from collections import defaultdict
from datetime import datetime

def rank_tags_by_volatility(file_path="posts.jsonl"):
    tag_counts = defaultdict(lambda: defaultdict(int))

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                continue

            record = post.get("record", {})
            created = record.get("createdAt")
            if not created:
                continue

            try:
                date = datetime.fromisoformat(created.replace("Z", "+00:00")).date().isoformat()
            except Exception:
                date = created[:10]

            for facet in record.get("facets", []) or []:
                for feat in facet.get("features", []) or []:
                    if feat.get("$type") == "app.bsky.richtext.facet#tag":
                        tag = feat.get("tag")
                        if tag:
                            tag_counts[tag.lower()][date] += 1

    df = pd.DataFrame(
        [{"tag": t, "date": d, "count": c} for t, dd in tag_counts.items() for d, c in dd.items()]
    )
    if df.empty:
        return pd.DataFrame(columns=["tag", "std_daily_freq"])

    df["date"] = pd.to_datetime(df["date"])
    all_days = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    out = []
    for tag, g in df.groupby("tag"):
        s = g.set_index("date")["count"].reindex(all_days, fill_value=0)
        vol = float(s.std(ddof=0))
        out.append({"tag": tag, "std_daily_freq": vol})

    result = pd.DataFrame(out).sort_values("std_daily_freq", ascending=False).reset_index(drop=True)
    pd.options.display.float_format = "{:.8f}".format  # print with more decimals
    return result


In [81]:
ranked_tags = rank_tags_by_volatility("posts.jsonl")
print(ranked_tags.head(20))



              tag  std_daily_freq
0           music    237.17925710
1          travel    197.05674310
2      technology    154.41321187
3        politics    136.41407552
4     photography    122.38071744
5        wellness    120.21414226
6         fashion    106.16590790
7            news    100.24090981
8             art     91.53228938
9          movies     91.48901573
10          style     84.10326985
11         nature     75.64496018
12     ballsacktv     64.40000000
13  ballsackradio     64.40000000
14           love     62.72607113
15        spotify     62.57347681
16          photo     60.61484967
17  photooftheday     58.91553276
18      beautiful     58.47939808
19         follow     58.42122902


In [82]:
import json
from datetime import datetime

def print_tag_dates(tag, file_path="posts.jsonl"):
    """
    Print all dates on which a given tag (case-insensitive)
    appears in the posts.jsonl file.
    """
    tag = tag.lower()
    dates = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                continue

            record = post.get("record", {})
            created = record.get("createdAt")
            if not created:
                continue

            try:
                date = datetime.fromisoformat(created.replace("Z", "+00:00")).date().isoformat()
            except Exception:
                date = created[:10]

            for facet in record.get("facets", []) or []:
                for feat in facet.get("features", []) or []:
                    if feat.get("$type") == "app.bsky.richtext.facet#tag":
                        t = feat.get("tag")
                        if t and t.lower() == tag:
                            dates.add(date)

    if not dates:
        print(f"No posts found with tag: {tag}")
        return []

    sorted_dates = sorted(dates)
    print(f"Tag '{tag}' was mentioned on {len(sorted_dates)} day(s):")
    for d in sorted_dates:
        print("  ", d)

    return sorted_dates
print_tag_dates("spotify")

Tag 'spotify' was mentioned on 3 day(s):
   2025-10-06
   2025-10-07
   2025-10-08


['2025-10-06', '2025-10-07', '2025-10-08']

In [78]:
import json
from datetime import datetime
from collections import Counter

def get_post_days(file_path="posts.jsonl", with_counts=False):
    dates = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                p = json.loads(line)
            except json.JSONDecodeError:
                continue

            created = (p.get("record") or {}).get("createdAt")
            if not created:
                continue

            # Safe ISO parsing (handles trailing 'Z')
            try:
                d = datetime.fromisoformat(created.replace("Z", "+00:00")).date().isoformat()
            except Exception:
                d = created[:10]  # fallback

            dates.append(d)

    if with_counts:
        return dict(sorted(Counter(dates).items()))  # {date: num_posts}
    return sorted(set(dates))  # ["YYYY-MM-DD", ...]
get_post_days()

['2025-10-04', '2025-10-05', '2025-10-06', '2025-10-07', '2025-10-08']